# Feature Engineering

---

## Objective

Transform transactional data into customer-product interaction features suitable for recommendation system modeling.

---

## Business Perspective

Recommendation systems require customer-product interaction information to understand purchasing behavior and recommend relevant products.

Accurate interaction features allow businesses to personalize customer experiences and improve cross-selling opportunities.

---

## Data Science Perspective

Transactional datasets must be transformed into structured interaction matrices before recommendation algorithms can be applied.

Feature engineering in recommendation systems focuses on representing customer preferences and product interactions.

---

## Methodology

The following feature engineering steps will be performed:

1. Create customer-product interaction dataset.
2. Aggregate purchase behavior.
3. Build interaction matrix.
4. Analyze matrix sparsity.
5. Prepare dataset for recommendation models.

---

In [1]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==================================================
# LOAD CLEAN DATASET
# ==================================================

PROJECT_ROOT = Path().resolve().parent

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "online_retail_clean.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (392692, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# ==================================================
# SELECT MODELING FEATURES
# ==================================================

interaction_df = df[
    [
        "CustomerID",
        "StockCode",
        "Description",
        "Quantity"
    ]
].copy()

print(interaction_df.shape)

display(interaction_df.head())

(392692, 4)


,CustomerID,StockCode,Description,Quantity
0,17850.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6
1,17850.0,71053,WHITE METAL LANTERN,6
2,17850.0,84406B,CREAM CUPID HEARTS COAT HANGER,8
3,17850.0,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6
4,17850.0,84029E,RED WOOLLY HOTTIE WHITE HEART.,6


## Interaction Definition

A customer-product interaction is defined as a successful purchase transaction.

The interaction strength is represented by the total quantity purchased by each customer for each product.

In [4]:
# ==================================================
# AGGREGATE INTERACTIONS
# ==================================================

interaction_df = (
    interaction_df
    .groupby(
        [
            "CustomerID",
            "StockCode",
            "Description"
        ]
    )["Quantity"]
    .sum()
    .reset_index()
)

interaction_df.rename(
    columns={
        "Quantity": "InteractionStrength"
    },
    inplace=True
)

print(
    "Aggregated Dataset Shape:",
    interaction_df.shape
)

display(interaction_df.head())

Aggregated Dataset Shape: (268398, 4)


,CustomerID,StockCode,Description,InteractionStrength
0,12346.0,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215
1,12347.0,16008,SMALL FOLDING SCISSOR(POINTED EDGE),24
2,12347.0,17021,NAMASTE SWAGAT INCENSE,36
3,12347.0,20665,RED RETROSPOT PURSE,6
4,12347.0,20719,WOODLAND CHARLOTTE BAG,40


In [5]:
# ==================================================
# INTERACTION STATISTICS
# ==================================================

print(
    "Unique Customers:",
    interaction_df["CustomerID"].nunique()
)

print(
    "Unique Products:",
    interaction_df["StockCode"].nunique()
)

print(
    "Customer-Product Interactions:",
    len(interaction_df)
)

Unique Customers: 4338
Unique Products: 3665
Customer-Product Interactions: 268398


In [6]:
# ==================================================
# CUSTOMER PRODUCT MATRIX
# ==================================================

customer_product_matrix = pd.pivot_table(

    interaction_df,

    index="CustomerID",

    columns="StockCode",

    values="InteractionStrength",

    fill_value=0

)

print(
    "Customer Product Matrix Shape:",
    customer_product_matrix.shape
)

customer_product_matrix.head()

Customer Product Matrix Shape: (4338, 3665)


StockCode   10002  10080  10120  10123C  10124A  10124G  10125  10133  10135  \
CustomerID                                                                     
12346.0       0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12347.0       0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12348.0       0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12349.0       0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12350.0       0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   

StockCode   11001  15030  15034  15036  15039  15044A  15044B  15044C  15044D  \
CustomerID                                                                      
12346.0       0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0     0.0   
12347.0       0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0     0.0   
12348.0       0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0     0.0   
12349.0       0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0     0.0   
12350.0       0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0     0.0   

StockCode   15056BL  15056N  15056P  15058A  15058B  15058C  15060B  16008  \
CustomerID                                                                   
12346.0         0.0     0.0     0.0     0.0     0.0     0.0     0.0    0.0   
12347.0         0.0     0.0     0.0     0.0     0.0     0.0     0.0   24.0   
12348.0         0.0     0.0     0.0     0.0     0.0     0.0     0.0    0.0   
12349.0         0.0     0.0     0.0     0.0     0.0     0.0     0.0    0.0   
12350.0         0.0     0.0     0.0     0.0     0.0     0.0     0.0    0.0   

StockCode   16010  16011  16012  16014  16015  16016  16020C  16033  16043  \
CustomerID                                                                   
12346.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0   
12347.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0   
12348.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0   
12349.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0   
12350.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0    0.0    0.0   

StockCode   16045  16046  16048  16049  16052  16054  16151A  16156L  16156S  \
CustomerID                                                                     
12346.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0   
12347.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0   
12348.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0   
12349.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0   
12350.0       0.0    0.0    0.0    0.0    0.0    0.0     0.0     0.0     0.0   

StockCode   16161G  16161M  16161P  16161U  16162L  16162M  16168M  16169E  \
CustomerID                                                                   
12346.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12347.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12348.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12349.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12350.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

StockCode   16169K  16169M  16169N  16169P  16202A  16202B  16202E  16206B  \
CustomerID                                                                   
12346.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12347.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12348.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12349.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
12350.0        0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   

StockCode   16207A  16207B  16216  16218  16219  16225  16235  16236  16237  \
CustomerID                                 

In [7]:
# ==================================================
# SPARSITY ANALYSIS
# ==================================================

matrix_values = customer_product_matrix.values

sparsity = 100 * (
    1.0 -
    np.count_nonzero(matrix_values)
    /
    matrix_values.size
)

print(
    f"Sparsity: {sparsity:.2f}%"
)

Sparsity: 98.32%


In [8]:
# ==================================================
# CUSTOMER ACTIVITY
# ==================================================

customer_activity = (
    interaction_df
    .groupby("CustomerID")
    .size()
)

display(
    customer_activity.describe()
)

count    4338.000000
mean       61.871369
std        86.288226
min         1.000000
25%        16.000000
50%        35.500000
75%        78.000000
max      1817.000000
dtype: float64

In [9]:
# ==================================================
# PRODUCT POPULARITY
# ==================================================

product_popularity = (
    interaction_df
    .groupby("StockCode")
    .size()
)

display(
    product_popularity.describe()
)

count    3665.000000
mean       73.232742
std        92.814049
min         1.000000
25%        11.000000
50%        40.000000
75%       100.000000
max       881.000000
dtype: float64

In [10]:
# ==================================================
# COLD START CUSTOMER ANALYSIS
# ==================================================

cold_start_customers = (
    customer_activity == 1
).sum()

cold_start_percentage = round(

    cold_start_customers
    /
    len(customer_activity)
    * 100,

    2
)

print(
    "Cold Start Customers:",
    cold_start_customers
)

print(
    "Percentage:",
    cold_start_percentage,
    "%"
)

Cold Start Customers: 90
Percentage: 2.07 %


# Findings

1. The interaction dataset contains 268,398 customer-product interactions.

2. The dataset includes 4,338 unique customers and 3,665 unique products.

3. The customer-product interaction matrix exhibits high sparsity (98.32%), which is typical for recommendation systems.

4. Customer purchasing behavior is highly heterogeneous, with some customers purchasing more than 1,800 products.

5. Product popularity is highly skewed, with several products purchased by hundreds of customers.

6. Only 2.07% of customers suffer from the cold-start problem.

7. The interaction dataset is suitable for collaborative filtering approaches.

---

# Business Perspective

1. Customer-product interaction data enables personalized recommendation strategies.

2. High interaction diversity provides opportunities for cross-selling and upselling.

3. The relatively low proportion of cold-start customers increases the feasibility of personalized recommendation systems.

4. Popular products can be leveraged for new customer recommendations.

5. Personalized recommendations are expected to improve customer engagement and increase average order value.

---

# Decision

1. Customer-product interactions will be used as the primary feature for recommendation modeling.

2. Popularity-based recommendation will be developed as the baseline model.

3. Collaborative filtering will be implemented as the advanced recommendation model.

4. The sparse interaction matrix will be preserved because sparsity is an inherent characteristic of recommendation datasets.

5. Cold-start customers will be addressed using popularity-based recommendations.

   

In [11]:
# ==================================================
# SAVE FEATURE DATASETS
# ==================================================

FEATURE_PATH = (
    PROJECT_ROOT /
    "data" /
    "features"
)

FEATURE_PATH.mkdir(
    parents=True,
    exist_ok=True
)

interaction_df.to_csv(

    FEATURE_PATH /
    "customer_product_interactions.csv",

    index=False
)

customer_product_matrix.to_csv(

    FEATURE_PATH /
    "customer_product_matrix.csv"
)

print(
    "Feature datasets saved successfully."
)

Feature datasets saved successfully.
